# M5 IA Agética - Agente de Atendimento ao Cliente

## 1. Introdução

Como Andrew explicou na aula, *planejamento com execução de código* significa permitir que o LLM **escreva código que se torna o próprio plano**.

Comparado a planos em texto simples ou baseados em JSON, essa abordagem é mais expressiva e flexível: o código não apenas documenta as etapas, mas também pode executá-las diretamente.

Neste laboratório, você implementará esse padrão de projeto na prática.

Em vez de pedir ao LLM para gerar um plano em formato JSON e, em seguida, executar manualmente cada etapa, permitiremos que ele **escreva código Python** que capture diretamente várias etapas de um plano. Ao executar esse código, podemos realizar consultas complexas automaticamente.

Para tornar as coisas concretas, simulamos uma **loja de óculos de sol** com um **estoque** de produtos e um conjunto de **transações** (vendas, devoluções, atualizações de saldo). Este exemplo mostra como o LLM pode gerar código para consultar ou atualizar registros, demonstrando a flexibilidade desse padrão.

### 1.1 Visão Geral do Laboratório
Iremos:
1. Criar conjuntos de dados simples de **estoque** e **transações**.

2. Construir um **bloco de esquema** descrevendo os dados.

3. Solicitar ao LLM que **escreva um plano como código Python** (com comentários explicando cada etapa).

4. Executar o código em um ambiente de teste para obter a resposta.

### 1.2 Resultados de Aprendizagem

Ao final deste laboratório, você será capaz de:

- **Explicar** por que permitir que o modelo escreva código (em vez de planos em JSON ou texto simples) possibilita um planejamento mais rico e flexível.

- **Solicitar** que um LLM produza código Python com comentários passo a passo que documentem e executem o plano.

- **Executar** o código gerado com segurança em um ambiente de teste e interpretar os resultados.

Isso ilustra como *Código como Ação* pode superar cadeias de ferramentas frágeis e abordagens de planejamento baseadas em JSON.

## 2. Setup

In [ ]:
# ==== Imports ====
from __future__ import annotations
import json
from dotenv import load_dotenv
from openai import OpenAI
import re, io, sys, traceback, json
from typing import Any, Dict, Optional
from tinydb import Query, where

# Utility modules
import utils      # helper functions for prompting/printing
import inv_utils  # functions for inventory, transactions, schema building, and TinyDB seeding

load_dotenv()
client = OpenAI()

No módulo `inv_utils`, temos funções como:

- `create_inventory()` – cria o inventário de óculos de sol.

- `create_transactions()` – cria o registro de transações inicial.

- `seed_db()` – carrega o inventário e as transações em um banco de dados com suporte a JSON.

- `build_schema_block()` – gera uma descrição do esquema usada no prompt.

- Funções auxiliares como `get_current_balance()` e `next_transaction_id()` – permitem que o LLM gerencie atualizações consistentes entre o inventário e as transações.

### 2.1 Criar Tabelas de Exemplo

Agora vamos criar duas pequenas tabelas para a simulação da loja de óculos de sol, usando o **[TinyDB](https://tinydb.readthedocs.io/)** — um banco de dados leve, orientado a documentos e escrito em Python puro.

O TinyDB armazena dados como documentos JSON e é ideal para pequenas aplicações ou protótipos, pois não requer configuração de servidor e permite consultar e atualizar dados facilmente.

As duas tabelas são:

- **`inventory_tbl`**: contém detalhes do produto, como nome, ID do item, descrição, quantidade em estoque e preço.

- **`transactions_tbl`**: começa com um saldo inicial e posteriormente registrará compras, devoluções e ajustes.

Você irá gerar essas tabelas usando funções auxiliares em `inv_utils` e, em seguida, visualizar as primeiras linhas abaixo.

In [ ]:
db, inventory_tbl, transactions_tbl = inv_utils.seed_db()

Agora, você pode inspecionar os registros em cada tabela, imprimindo-os como JSON formatado:

In [ ]:
utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table")
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table")

Como você pode ver acima, os esquemas de cada tabela são os seguintes:

<div style="border:1px solid #BFDBFE; border-left:6px solid #3B82F6; background:#EFF6FF; border-radius:6px; padding:16px; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif; line-height:1.6; color:#1E3A8A;">

<h4 style="margin-top:0; color:#1E40AF;">Tabela de Inventário (<code>inventory_tbl</code>)</h4>

<ul>
<li><strong>item_id</strong> (string): Identificador único do produto (ex.: SG001).</li>

<li><strong>name</strong> (string): Estilo dos óculos de sol (ex.: Aviador, <li><strong>descrição</strong> (string): Descrição textual do produto.</li>
<li><strong>quantidade_em_estoque</strong> (int): Estoque atual disponível.</li>
<li><strong>preço</strong> (float): Preço em USD.</li>
</ul>
<h4 style="margin-top:1em; color:#1E40AF;">Tabela de Transações (<code>transactions_tbl</code>)</h4>
<ul>
<li><strong>id_da_transação</strong> (string): Identificador único (ex.: TXN001).</li>
<li><strong>nome_do_cliente</strong> (string): Nome do cliente ou <code>SALDO_ABERTURA</code> para entrada inicial.</li>
<li><strong>resumo_da_transação</strong> (string): Breve descrição da transação.</li>
<li><strong>valor_da_transação</strong> <li><strong>balance_after_transaction</strong> (float): Saldo atual após a aplicação da transação.</li>
<li><strong>timestamp</strong> (string): Data e hora da transação no formato ISO-8601.</li>
</ul>
</div>

## Planejamento com Execução de Código

### 2.1. O plano

Assim que o esquema estiver claro, você criará o **prompt** que instrui o modelo a *planejar escrevendo código* e, em seguida, executar esse código. Como Andrew enfatizou, o código é o plano: o modelo explica cada etapa em comentários e, em seguida, a executa. O prompt abaixo também faz com que o modelo decida por si só se a solicitação é somente leitura ou uma mudança de estado e impõe uma execução segura (sem E/S, sem rede, somente consultas TinyDB, mutações consistentes).

In [ ]:
PROMPT = """You are a senior data assistant. PLAN BY WRITING PYTHON CODE USING TINYDB.

Database Schema & Samples (read-only):
{schema_block}

Execution Environment (already imported/provided):
- Variables: db, inventory_tbl, transactions_tbl  # TinyDB Table objects
- Helpers: get_current_balance(tbl) -> float, next_transaction_id(tbl, prefix="TXN") -> str
- Natural language: user_request: str  # the original user message

PLANNING RULES (critical):
- Derive ALL filters/parameters from user_request (shape/keywords, price ranges "under/over/between", stock mentions,
  quantities, buy/return intent). Do NOT hard-code values.
- Build TinyDB queries dynamically with Query(). If a constraint isn't in user_request, don't apply it.
- Be conservative: if intent is ambiguous, do read-only (DRY RUN).

TRANSACTION POLICY (hard):
- Do NOT create aggregated multi-item transactions.
- If the request contains multiple items, create a separate transaction row PER ITEM.
- For each item:
  - compute its own line total (unit_price * qty),
  - insert ONE transaction with that amount,
  - update balance sequentially (balance += line_total),
  - update the item’s stock.
- If any requested item lacks sufficient stock, do NOT mutate anything; reply with STATUS="insufficient_stock".

HUMAN RESPONSE REQUIREMENT (hard):
- You MUST set a variable named `answer_text` (type str) with a short, customer-friendly sentence (1–2 lines).
- This sentence is the only user-facing message. No dataframes/JSON, no boilerplate disclaimers.
- If nothing matches, politely say so and offer a nearby alternative (closest style/price) or a next step.

ACTION POLICY:
- If the request clearly asks to change state (buy/purchase/return/restock/adjust):
    ACTION="mutate"; SHOULD_MUTATE=True; perform the change and write a matching transaction row.
  Otherwise:
    ACTION="read"; SHOULD_MUTATE=False; simulate and explain briefly as a dry run (in logs only).

FAILURE & EDGE-CASE HANDLING (must implement):
- Do not capture outer variables in Query.test. Pass them as explicit args.
- Always set a short `answer_text`. Also set a string `STATUS` to one of:
  "success", "no_match", "insufficient_stock", "invalid_request", "unsupported_intent".
- no_match: No items satisfy the filters → suggest the closest in style/price, or invite a different range.
- insufficient_stock: Item found but stock < requested qty → state available qty and offer the max you can fulfill.
- invalid_request: Unable to parse essential info (e.g., quantity for a purchase/return) → ask for the missing piece succinctly.
- unsupported_intent: The action is outside the store’s capabilities → provide the nearest supported alternative.
- In all cases, keep the tone helpful and concise (1–2 sentences). Put technical details (e.g., ACTION/DRY RUN) only in stdout logs.

OUTPUT CONTRACT:
- Return ONLY executable Python between these tags (no extra text):
  <execute_python>
  # your python
  </execute_python>

CODE CHECKLIST (follow in code):
1) Parse intent & constraints from user_request (regex ok).
2) Build TinyDB condition incrementally; query inventory_tbl.
3) If mutate: validate stock, update inventory, insert a transaction (new id, amount, balance, timestamp).
4) ALWAYS set:
   - `answer_text` (human sentence, required),
   - `STATUS` (see list above).
   Also print a brief log to stdout, e.g., "LOG: ACTION=read DRY_RUN=True STATUS=no_match".
5) Optional: set `answer_rows` or `answer_json` if useful, but `answer_text` is mandatory.

TONE EXAMPLES (for `answer_text`):
- success: "Yes, we have our Classic sunglasses, a round frame, for $60."
- no_match: "We don’t have round frames under $100 in stock right now, but our Moon round frame is available at $120."
- insufficient_stock: "We only have 1 pair of Classic left; I can reserve that for you."
- invalid_request: "I can help with that—how many pairs would you like to purchase?"
- unsupported_intent: "We can’t refurbish frames, but I can suggest similar new models."

Constraints:
- Use TinyDB Query for filtering. Standard library imports only if needed.
- Keep code clear and commented with numbered steps.

User request:
{question}
"""


### 2.2 Do Prompt ao Código (Planejamento em Código)

Vamos gerar um código que **é o plano**.

Em vez de pedir ao modelo para gerar um plano em JSON e executá-lo passo a passo com várias ferramentas pequenas, vamos fazer com que ele **escreva um código Python que codifique todo o plano** (por exemplo, “filtre isto, depois calcule aquilo e, em seguida, atualize esta linha”). A função `generate_llm_code`:

1. **Constrói um esquema dinâmico** a partir de `inventory_tbl` e `transactions_tbl` para que o modelo veja campos, tipos e exemplos reais.

2. **Formata o prompt** com esse esquema mais a pergunta do usuário.

3. **Chama o modelo** para produzir uma resposta de **plano com código** — normalmente um bloco `<execute_python>...</execute_python>` cujo corpo contém a lógica passo a passo.

4. **Retorna a resposta completa** (incluindo o plano e o código).

*Não executamos nada nesta etapa.*

Por que esse padrão? Vamos aproveitar o Python/TinyDB como um conjunto de ferramentas robusto que o modelo já "conhece", para que ele possa compor soluções de várias etapas diretamente no código, em vez de depender de um conjunto crescente de ferramentas específicas. Extrairemos e executaremos o código em uma etapa posterior.

In [ ]:
# ---------- 1) Code generation ----------
def generate_llm_code(
    prompt: str,
    *,
    inventory_tbl,
    transactions_tbl,
    model: str = "gpt-4.1-mini",
    temperature: float = 0.2,
) -> str:
    """
    Ask the LLM to produce a plan-with-code response.
    Returns the FULL assistant content (including surrounding text and tags).
    The actual code extraction happens later in execute_generated_code.
    """
    schema_block = inv_utils.build_schema_block(inventory_tbl, transactions_tbl)
    prompt = PROMPT.format(schema_block=schema_block, question=prompt)

    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {
                "role": "system",
                "content": "You write safe, well-commented TinyDB code to handle data questions and updates."
            },
            {"role": "user", "content": prompt},
        ],
    )
    content = resp.choices[0].message.content or ""
    
    return content  

### 2.3 Experimente um Exemplo de Pergunta (Planejamento em Código)

Usaremos a mesma pergunta que Andrew usou na aula:

> **Pergunta:** “Você tem óculos de sol redondos em estoque por menos de US$ 100?”

Antes de gerar qualquer código, vamos inspecionar manualmente as tabelas do TinyDB para ver se existem armações realmente *redondas* (correspondência apenas por palavras) e quais são seus preços. Execute a próxima célula para visualizar o estoque e destacar os itens que correspondem ao filtro de palavras “redondo”.

In [ ]:
Item = Query()                    # Create a Query object to reference fields (e.g., Item.name, Item.description)

# Search the inventory table for documents where either the description OR the name
# contains the word "round" (case-insensitive). The check is done inline:
# - (v or "") ensures we handle None by converting it to an empty string
# - .lower() normalizes case
# - " round " enforces a crude word boundary (won't match "wraparound")
round_sunglasses = inventory_tbl.search(
    (Item.description.test(lambda v: " round " in ((v or "").lower()))) |
    (Item.name.test(        lambda v: " round " in ((v or "").lower())))
)

# Render the results as formatted JSON in the notebook UI
utils.print_html(json.dumps(round_sunglasses, indent=2), title="Inventory Status: Round Sunglasses")

Ótimo! Temos armações redondas disponíveis. Após inspeção manual, constatamos que temos dois modelos redondos em estoque, mas apenas **um** custa **menos de $100**. Portanto, o item que atende ao requisito é:

````python
{
  "item_id": "SG005",
  "name": "Classic",
  "description": "Classic round profile with minimalist metal frames, offering a timeless and versatile style that fits both casual and formal wear.",
  "quantity_in_stock": 10,
  "price": 60
}
````

Agora, vamos pedir ao modelo para **gerar um plano em código** que responda à solicitação de Andrew (ainda sem execução).

In [ ]:
# Andrew's prompt from the lecture
prompt_round = "Do you have any round sunglasses in stock that are under $100?"

# Generate the plan-as-code (FULL content; may include <execute_python> tags)
full_content_round = generate_llm_code(
    prompt_round,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1.0,
)

# Inspect the LLM’s plan + code (no execution here)
utils.print_html(full_content_round, title="Plan with Code (Full Response)")

### 2.4. Defina a função executora (execute um plano fornecido)

Agora, definiremos a função que **recebe um plano gerado pelo modelo e o executa** com segurança:

- Ela **aceita** a resposta completa do LLM (com `<execute_python>…</execute_python>`) **ou** código Python puro.

- Ela **extrai** o bloco executável quando necessário.

- Ela executa o código em um **namespace controlado** (somente tabelas TinyDB + helpers seguros).

- Ela captura **stdout**, **erros** e as variáveis ​​de resposta definidas pelo modelo (`answer_text`, `answer_rows`, `answer_json`).

- Ela renderiza snapshots da tabela **antes/depois** para explicitar os efeitos colaterais.

Esta é a “executora” que transforma um **plano como código** em ações e uma resposta concisa para o usuário.

In [ ]:
# --- Helper: extract code between <execute_python>...</execute_python> ---
def _extract_execute_block(text: str) -> str:
    """
    Returns the Python code inside <execute_python>...</execute_python>.
    If no tags are found, assumes 'text' is already raw Python code.
    """
    if not text:
        raise RuntimeError("Empty content passed to code executor.")
    m = re.search(r"<execute_python>(.*?)</execute_python>", text, re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()


# ---------- 2) Code execution ----------
def execute_generated_code(
    code_or_content: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    user_request: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Execute code in a controlled namespace.
    Accepts either raw Python code OR full content with <execute_python> tags.
    Returns minimal artifacts: stdout, error, and extracted answer.
    """
    # Extract code here (now centralized)
    code = _extract_execute_block(code_or_content)

    SAFE_GLOBALS = {
        "Query": Query,
        "get_current_balance": inv_utils.get_current_balance,
        "next_transaction_id": inv_utils.next_transaction_id,
        "user_request": user_request or "",
    }
    SAFE_LOCALS = {
        "db": db,
        "inventory_tbl": inventory_tbl,
        "transactions_tbl": transactions_tbl,
    }

    # Capture stdout from the executed code
    _stdout_buf, _old_stdout = io.StringIO(), sys.stdout
    sys.stdout = _stdout_buf
    err_text = None
    try:
        exec(code, SAFE_GLOBALS, SAFE_LOCALS)
    except Exception:
        err_text = traceback.format_exc()
    finally:
        sys.stdout = _old_stdout
    printed = _stdout_buf.getvalue().strip()

    # Extract possible answers set by the generated code
    answer = (
        SAFE_LOCALS.get("answer_text")
        or SAFE_LOCALS.get("answer_rows")
        or SAFE_LOCALS.get("answer_json")
    )


    return {
        "code": code,            # <- ya sin etiquetas
        "stdout": printed,
        "error": err_text,
        "answer": answer,
        "transactions_tbl": transactions_tbl.all(),  # For inspection
        "inventory_tbl": inventory_tbl.all(),  # For inspection
    }

Você conferiu as prateleiras e confirmou que existe exatamente um modelo redondo por menos de US$ 100. Agora vem a parte divertida: vamos entregar o plano do modelo em código ao nosso executor e observar o trabalho. O executor irá extrair o bloco `<code><execute_python>...</execute_python></code>`, executá-lo em um ambiente isolado e, em seguida, mostrar tudo o que importa — o que mudou nas tabelas (antes/depois), quaisquer registros que o plano tenha impresso e a resposta final, amigável para o cliente.

In [ ]:
# Execute the generated plan for the round-sunglasses question
result = execute_generated_code(
    full_content_round,          # the full LLM response you generated earlier
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_round, # e.g., "Do you have any round sunglasses in stock that are under $100?"
)

# Peek at exactly what Python the plan executed
utils.print_html(result["answer"], title="Plan Execution · Extracted Answer")

Como podem ver, este é o resultado esperado com base em nossa análise manual anterior.

## 2.4 Devolver Dois Óculos de Sol Aviador

Na etapa anterior, apenas **consultamos** os dados, portanto, o estoque e as transações permaneceram inalterados.

Agora, vamos lidar com um cenário de **devolução** usando o padrão de planejamento em código:
> **Solicitação:** “Devolver 2 óculos de sol Aviador que comprei na semana passada.”

Antes de gerar o plano, vamos **inspecionar o estoque atual** do modelo *Aviador*.

In [ ]:
Item = Query()                    # Create a Query object to reference fields (e.g., Item.name, Item.description)

# Query: fetch all inventory rows whose 'name' is exactly "Aviator".
# Notes:
# - This is a case-sensitive equality check. "aviator" won't match.
# - If you need case-insensitive matching, consider a .test(...) or .matches(...) with re.I.
aviators = inventory_tbl.search(
    (Item.name == "Aviator")
)

# Display the matched documents in a readable JSON panel
utils.print_html(json.dumps(aviators, indent=2), title="Inventory status: Aviator sunglasses before return")

O inventário confirma a disponibilidade de um SKU do Aviator — **SG001 (Aviator)**: **23** unidades a **$80** cada. Agora, vamos gerar um plano para responder à pergunta:

In [ ]:
prompt_aviator = "Return 2 Aviator sunglasses I bought last week."

# Generate the plan-as-code (FULL content; may include <execute_python> tags)
full_content_aviator = generate_llm_code(
    prompt_aviator,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1,
)

# Inspect the LLM’s plan + code (no execution here)
utils.print_html(full_content_aviator, title="Plan with Code (Full Response)")

Antes de executarmos o plano, vamos verificar o status atual das transações.

In [ ]:
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table Before Return")

O registro de transações mostra atualmente uma única entrada — o saldo inicial (`TXN001`) de `$500,00` registrado em `2025-10-03T09:16:59.628898`.

Pronto para executar o plano? Clique na célula abaixo.

In [ ]:
# Execute the generated plan for the round-sunglasses question
result = execute_generated_code(
    full_content_aviator,          # the full LLM response you generated earlier
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    user_request=prompt_aviator, # e.g., "Return 2 aviator sunglasses I bought last week."
)

# Peek at exactly what Python the plan executed
utils.print_html(result["answer"], title="Plan Execution · Extracted Answer")

Você pode ver abaixo que uma nova transação foi inserida para a devolução dos óculos de sol Aviator.

In [ ]:
utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table After Return")

E ao executar a célula abaixo, você verá o estoque do Aviator aumentar para 25 (`quantity_in_stock`).

In [ ]:
Item = Query()                  

aviators = inventory_tbl.search(
    (Item.name == "Aviator")
)

utils.print_html(json.dumps(aviators, indent=2), title="Inventory status: Aviator sunglasses after return")

## 3. Unindo Tudo: Agente de Atendimento ao Cliente

Você construiu as peças — esquema, prompt, gerador de código e executor. Agora, vamos integrá-las em um único auxiliar que recebe uma solicitação em linguagem natural, gera um plano como código, o executa com segurança e exibe o resultado (além de tabelas de antes e depois).

**O que este agente faz**
- Opcionalmente, reinicializa os dados de demonstração para uma execução limpa.

- Gera o plano (Python dentro de `<execute_python>…</execute_python>`).

- Executa o plano em um namespace controlado (TinyDB + auxiliares).

- Exibe um `answer_text` conciso e renderiza instantâneos de antes e depois.

In [ ]:
def customer_service_agent(
    question: str,
    *,
    db,
    inventory_tbl,
    transactions_tbl,
    model: str = "o4-mini",
    temperature: float = 1.0,
    reseed: bool = False,
) -> dict:
    """
    End-to-end helper:
      1) (Optional) reseed inventory & transactions
      2) Generate plan-as-code from `question`
      3) Execute in a controlled namespace
      4) Render before/after snapshots and return artifacts

    Returns:
      {
        "full_content": <raw LLM response (may include <execute_python> tags)>,
        "exec": {
            "code": <extracted python>,
            "stdout": <plan logs>,
            "error": <traceback or None>,
            "answer": <answer_text/rows/json>,
            "inventory_after": [...],
            "transactions_after": [...]
        }
      }
    """
    # 0) Optional reseed
    if reseed:
        inv_utils.create_inventory()
        inv_utils.create_transactions()

    # 1) Show the question
    utils.print_html(question, title="User Question")

    # 2) Generate plan-as-code (FULL content)
    full_content = generate_llm_code(
        question,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        model=model,
        temperature=temperature,
    )
    utils.print_html(full_content, title="Plan with Code (Full Response)")

    # 3) Before snapshots
    utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table · Before")
    utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table · Before")

    # 4) Execute
    exec_res = execute_generated_code(
        full_content,
        db=db,
        inventory_tbl=inventory_tbl,
        transactions_tbl=transactions_tbl,
        user_request=question,
    )

    # 5) After snapshots + final answer
    utils.print_html(exec_res["answer"], title="Plan Execution · Extracted Answer")
    utils.print_html(json.dumps(inventory_tbl.all(), indent=2), title="Inventory Table · After")
    utils.print_html(json.dumps(transactions_tbl.all(), indent=2), title="Transactions Table · After")

    # 6) Return artifacts
    return {
        "full_content": full_content,
        "exec": {
            "code": exec_res["code"],
            "stdout": exec_res["stdout"],
            "error": exec_res["error"],
            "answer": exec_res["answer"],
            "inventory_after": inventory_tbl.all(),
            "transactions_after": transactions_tbl.all(),
        },
    }


## 4. Experimente (com o Agente de Atendimento ao Cliente)

Use a função auxiliar `customer_service_agent(...)` para ir de uma solicitação em linguagem natural → planejamento como código → execução segura → capturas de tela antes e depois.

**Experimente estas solicitações:**
1) **Somente leitura (exemplo do Andrew):**

“Vocês têm óculos de sol redondos em estoque por menos de US$ 100?”

2) **Mutação — retorno:**

“Devolver 2 óculos de sol Aviador.”

3) **Mutação — compra:**

“Comprar 3 óculos de sol Wayfarer para a cliente Alice.”

4) **Mutação — compra de vários itens:**

“Quero comprar 3 pares de óculos de sol clássicos e 1 par de óculos de sol Aviador.”

<div style="border:1px solid #93c5fd; border-left:6px solid #3b82f6; background:#eff6ff; border-radius:8px; padding:14px 16px; color:#1e3a8a; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">
🔎 <strong>O que <code>reseed=True</code> faz?</strong><br><br>

Quando você chama <code>customer_service_agent(..., reseed=True)</code>, o agente <em>reinicializa</em> os dados de demonstração antes de executar seu prompt:
<ul style="margin:8px 0 0 18px;">

<li><strong>Redefine</strong> a <code>inventory_tbl</code> para o Conjunto de produtos padrão.</li>
<li><strong>Redefine</strong> a tabela <code>transactions_tbl</code> para um único lançamento de saldo inicial.</li>
<li>Garante uma execução <strong>limpa e reproduzível</strong> para que os resultados não sejam afetados por testes anteriores.</li>

</ul>

Defina <code>reseed=False</code> se desejar <strong>preservar</strong> o estado atual e continuar de operações anteriores.

</div>

In [ ]:
prompt = "I want to buy 3 pairs of classic sunglasses and 1 pair of aviator sunglasses."

out = customer_service_agent(
    prompt,
    db=db,
    inventory_tbl=inventory_tbl,
    transactions_tbl=transactions_tbl,
    model="o4-mini",
    temperature=1.0,
    reseed=True,   # set False to keep current state of the inventory and the transactions
)

## 5. Principais Conclusões

- **Você deixou o código ser o plano.** Seguindo a ideia de "código como ação" de Andrew, você fez o modelo escrever código Python que encadeia as etapas (filtrar → computar → atualizar) e então você simplesmente o executou.

- **Você evitou a sopa de ferramentas frágeis.** Em vez de acumular pequenas ferramentas ou planos JSON, você usou Python/TinyDB — fornecendo ao modelo um conjunto de ferramentas amplo e familiar que lida com diversos formatos de consulta com um único comando.

- **Você manteve as execuções seguras e visíveis.** Você executou em um namespace controlado, capturou logs/erros e revisou as tabelas antes e depois — assim você sempre sabe o que mudou e por quê.

<div style="border:1px solid #22c55e; border-left:6px solid #16a34a; background:#dcfce7; border-radius:6px; padding:14px 16px; color:#064e3b; font-family:system-ui,-apple-system,Segoe UI,Roboto,Ubuntu,Cantarell,Noto Sans,sans-serif;">

🎉 <strong>Parabéns!</strong>

Você acaba de concluir o laboratório e construiu um fluxo de trabalho de atendimento ao cliente <em>agente</em>. Você permitiu que o modelo escrevesse o código conforme o planejado, executou-o com segurança e usou validações simples para manter as atualizações confiáveis. Quando algo falhou, você apresentou razões claras e fáceis de entender; quando tudo funcionou, você viu exatamente o que mudou por meio de capturas de tela de antes e depois.

Com esse padrão — planejamento em código, mais execução transparente — você está pronto para criar seus próprios fluxos de trabalho que parecem automáticos, seguros e fáceis de estender. 🚀

</div>